In [9]:
#2.11
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ==========================================
# BƯỚC 1: KHAI BÁO BIẾN ĐẦU VÀO VÀ ĐẦU RA
# ==========================================
# Antecedents (Đầu vào)
distance = ctrl.Antecedent(np.arange(0, 51, 1), 'distance')   # 0 đến 50 km
traffic = ctrl.Antecedent(np.arange(0, 101, 1), 'traffic')    # Mức độ tắc nghẽn 0-100%
weather = ctrl.Antecedent(np.arange(0, 11, 1), 'weather')     # Mức độ thời tiết xấu 0-10 (0 là đẹp, 10 là bão)

# Consequent (Đầu ra)
price_multiplier = ctrl.Consequent(np.arange(1.0, 3.1, 0.1), 'price_multiplier') # Hệ số nhân giá từ 1.0 đến 3.0

# ==========================================
# BƯỚC 2: ĐỊNH NGHĨA TẬP MỜ (Membership Functions)
# ==========================================
# Dựa theo dữ liệu trang 86 của giáo trình

# Quãng đường
distance['ngan'] = fuzz.trapmf(distance.universe, [0, 0, 3, 5])
distance['trung_binh'] = fuzz.trapmf(distance.universe, [2, 4, 8, 10])
distance['dai'] = fuzz.trapmf(distance.universe, [6, 10, 20, 25])
distance['rat_xa'] = fuzz.trapmf(distance.universe, [15, 25, 50, 50])

# Tình trạng giao thông
traffic['thap'] = fuzz.trimf(traffic.universe, [0, 0, 30])
traffic['trung_binh'] = fuzz.trimf(traffic.universe, [20, 50, 70])
traffic['cao'] = fuzz.trimf(traffic.universe, [60, 100, 100])

# Điều kiện khí hậu (Thời tiết)
weather['tot'] = fuzz.trimf(weather.universe, [0, 0, 4])
weather['trung_binh'] = fuzz.trimf(weather.universe, [3, 5, 7])
weather['xau'] = fuzz.trimf(weather.universe, [6, 10, 10])

# Hệ số nhân giá
price_multiplier['gia_thap'] = fuzz.trimf(price_multiplier.universe, [1.0, 1.0, 1.3])
price_multiplier['gia_trung_binh'] = fuzz.trimf(price_multiplier.universe, [1.2, 1.5, 1.8])
price_multiplier['gia_cao'] = fuzz.trimf(price_multiplier.universe, [1.6, 2.0, 2.4])
price_multiplier['gia_rat_cao'] = fuzz.trapmf(price_multiplier.universe, [2.2, 2.6, 3.0, 3.0])

# ==========================================
# BƯỚC 3: THIẾT LẬP LUẬT MỜ (Fuzzy Rules)
# ==========================================
# Chọn lọc một vài luật điển hình từ giáo trình

# Luật 1: (Khoảng cách ngắn) VÀ (Lưu lượng thấp) -> Giá thấp
rule1 = ctrl.Rule(distance['ngan'] & traffic['thap'], price_multiplier['gia_thap'])

# Luật 3: (Khoảng cách trung bình) VÀ (Lưu lượng cao) -> Giá cao
rule3 = ctrl.Rule(distance['trung_binh'] & traffic['cao'], price_multiplier['gia_cao'])

# Luật 6: (Khoảng cách rất xa) VÀ (Lưu lượng cao) VÀ (Thời tiết xấu) -> Giá rất cao
rule6 = ctrl.Rule(distance['rat_xa'] & traffic['cao'] & weather['xau'], price_multiplier['gia_rat_cao'])

# Luật 10: (Khoảng cách trung bình) VÀ (Lưu lượng trung bình) VÀ (Thời tiết vừa phải) -> Giá trung bình
rule10 = ctrl.Rule(distance['trung_binh'] & traffic['trung_binh'] & weather['trung_binh'], price_multiplier['gia_trung_binh'])

# Khởi tạo hệ thống kiểm soát
pricing_ctrl = ctrl.ControlSystem([rule1, rule3, rule6, rule10])
pricing_sim = ctrl.ControlSystemSimulation(pricing_ctrl)

# ==========================================
# BƯỚC 4: CHẠY MÔ PHỎNG VỚI DỮ LIỆU CỤ THỂ
# ==========================================
# Giả sử: Bạn đi từ ĐH Kinh tế TP.HCM (Quận 10) về nhà trọ
input_distance = 6.5  # km
input_traffic = 85    # % kẹt xe (giờ tan tầm)
input_weather = 8     # Mưa to

print("--- THÔNG SỐ CHUYẾN ĐI ---")
print(f"Quãng đường: {input_distance} km")
print(f"Mức độ kẹt xe: {input_traffic}%")
print(f"Mức độ thời tiết xấu: {input_weather}/10")

# Truyền tham số
pricing_sim.input['distance'] = input_distance
pricing_sim.input['traffic'] = input_traffic
pricing_sim.input['weather'] = input_weather

# Tính toán
pricing_sim.compute()

# Kết quả
he_so = pricing_sim.output['price_multiplier']

# Tính thử giá tiền (Giả sử giá gốc 5500d/km)
gia_goc = input_distance * 5500
gia_cuoi = gia_goc * he_so

print("\n--- KẾT QUẢ ĐỊNH GIÁ GRAB-BIKE ---")
print(f"Hệ số nhân (Surge Pricing): x{he_so:.2f}")
print(f"Giá cuốc xe dự kiến: {gia_cuoi:,.0f} VNĐ (so với giá gốc {gia_goc:,.0f} VNĐ)")

# (Tùy chọn) Hiển thị đồ thị trong Colab
# price_multiplier.view(sim=pricing_sim)

--- THÔNG SỐ CHUYẾN ĐI ---
Quãng đường: 6.5 km
Mức độ kẹt xe: 85%
Mức độ thời tiết xấu: 8/10

--- KẾT QUẢ ĐỊNH GIÁ GRAB-BIKE ---
Hệ số nhân (Surge Pricing): x2.00
Giá cuốc xe dự kiến: 71,500 VNĐ (so với giá gốc 35,750 VNĐ)


In [10]:
#2.12
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ==========================================
# BƯỚC 1: KHAI BÁO BIẾN ĐẦU VÀO VÀ ĐẦU RA (Trang 89)
# ==========================================
# Đầu vào
store_rating = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'store_rating') # Xếp hạng sao (0 - 5.0)
sales_volume = ctrl.Antecedent(np.arange(0, 101, 1), 'sales_volume')   # Khối lượng bán hàng (Quy đổi 0-100 điểm)
profit_margin = ctrl.Antecedent(np.arange(0, 101, 1), 'profit_margin') # Biên lợi nhuận (0-100%)
seasonality = ctrl.Antecedent(np.arange(0, 11, 1), 'seasonality')      # Sự kiện mùa vụ (0: Không có, 10: Sale lớn như 11/11)
competitor = ctrl.Antecedent(np.arange(0, 101, 1), 'competitor')       # Đối thủ giảm giá (0-100%)

# Đầu ra
discount = ctrl.Consequent(np.arange(0, 71, 1), 'discount') # Tỷ lệ chiết khấu (0 - 70%)

# ==========================================
# BƯỚC 2: ĐỊNH NGHĨA TẬP MỜ (Theo mô tả trang 89)
# ==========================================
# Đánh giá cửa hàng
store_rating['thap'] = fuzz.trapmf(store_rating.universe, [0, 0, 3.8, 4.0])
store_rating['tb'] = fuzz.trimf(store_rating.universe, [3.9, 4.25, 4.6])
store_rating['cao'] = fuzz.trapmf(store_rating.universe, [4.5, 4.8, 5.0, 5.0])

# Tỷ lệ phần trăm chiết khấu (Đầu ra)
discount['rat_thap'] = fuzz.trapmf(discount.universe, [0, 0, 3, 5])
discount['thap'] = fuzz.trapmf(discount.universe, [4, 7, 8, 10])
discount['tb'] = fuzz.trapmf(discount.universe, [9, 14, 16, 20])
discount['cao'] = fuzz.trapmf(discount.universe, [18, 28, 32, 40])
discount['rat_cao'] = fuzz.trapmf(discount.universe, [35, 50, 70, 70])

# (Để code ngắn gọn gọn, mình dùng automf / trimf chuẩn cho các biến còn lại)
sales_volume.automf(names=['thap', 'tb', 'cao'])
profit_margin.automf(names=['thap', 'tb', 'cao'])
seasonality.automf(names=['khong_co', 'tb', 'cao'])
competitor.automf(names=['thap', 'tb', 'cao'])

# ==========================================
# BƯỚC 3: THIẾT LẬP LUẬT MỜ (Trang 89 & 90)
# ==========================================
# 1. Shop uy tín, bán chạy, lãi dày -> Giữ giá, chiết khấu rất thấp
rule1 = ctrl.Rule(store_rating['cao'] & sales_volume['cao'] & profit_margin['cao'], discount['rat_thap'])

# 2. Shop mới/thấp, ế ẩm nhưng lãi cao -> Đẩy mạnh chiết khấu cao để kéo số
rule2 = ctrl.Rule(store_rating['thap'] & sales_volume['thap'] & profit_margin['cao'], discount['cao'])

# 3. Ngày hội Sale lớn (11/11) VÀ đối thủ khô máu -> Bắt buộc chiết khấu rất cao
rule3 = ctrl.Rule(seasonality['cao'] & competitor['cao'], discount['rat_cao'])

# 4. Mọi thứ bình thường -> Chiết khấu trung bình kích cầu nhẹ
rule4 = ctrl.Rule(store_rating['tb'] & sales_volume['tb'] & profit_margin['tb'], discount['tb'])

# 7. Khối lượng bán thấp VÀ Biên lợi nhuận thấp -> Không được giảm giá sâu, chiết khấu rất thấp để tránh lỗ
rule7 = ctrl.Rule(sales_volume['thap'] & profit_margin['thap'], discount['rat_thap'])

# Khởi tạo Engine
shopee_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule7])
shopee_sim = ctrl.ControlSystemSimulation(shopee_ctrl)

# ==========================================
# BƯỚC 4: CHẠY MÔ PHỎNG VỚI DỮ LIỆU TỪ TRANG 90
# ==========================================
# Đề bài trang 90: Xếp hạng: 4.3 (TB), Khối lượng: TB, Lợi nhuận: Thấp, Sự kiện: Cao (9.9), Đối thủ: Cao.
print("--- THÔNG SỐ CỬA HÀNG SHOPEE ---")
print("Đánh giá sao: 4.3 (Trung bình)")
print("Sự kiện: 9/10 (Cao - Ngày hội Mega Sale)")
print("Khuyến mãi của đối thủ: 80% (Cao)")

shopee_sim.input['store_rating'] = 4.3
shopee_sim.input['sales_volume'] = 50   # Mức trung bình
shopee_sim.input['profit_margin'] = 15  # Mức thấp (15%)
shopee_sim.input['seasonality'] = 9     # Sự kiện cao
shopee_sim.input['competitor'] = 80     # Đối thủ cạnh tranh gắt

# Tính toán
shopee_sim.compute()

# Kết quả
muc_chiet_khau = shopee_sim.output['discount']

print(f"\n--- KẾT QUẢ ĐỀ XUẤT TỪ HỆ THỐNG MỜ ---")
print(f"Mức chiết khấu tối ưu: {muc_chiet_khau:.1f}%")

# So sánh với giá trị minh họa (Sản phẩm gốc 100.000đ)
gia_goc = 100000
gia_sau_giam = gia_goc * (1 - muc_chiet_khau/100)
print(f"Giá bán sau chiết khấu (áp dụng cho SP 100k): {gia_sau_giam:,.0f} VNĐ")

--- THÔNG SỐ CỬA HÀNG SHOPEE ---
Đánh giá sao: 4.3 (Trung bình)
Sự kiện: 9/10 (Cao - Ngày hội Mega Sale)
Khuyến mãi của đối thủ: 80% (Cao)

--- KẾT QUẢ ĐỀ XUẤT TỪ HỆ THỐNG MỜ ---
Mức chiết khấu tối ưu: 49.2%
Giá bán sau chiết khấu (áp dụng cho SP 100k): 50,833 VNĐ


In [15]:
#2.13
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# 1. Khai báo biến
product_demand = ctrl.Antecedent(np.arange(0, 101, 1), 'product_demand')
comp_pressure = ctrl.Antecedent(np.arange(0, 101, 1), 'comp_pressure')
store_rep = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'store_rep')
profit_margin = ctrl.Antecedent(np.arange(0, 101, 1), 'profit_margin')
seasonal_demand = ctrl.Antecedent(np.arange(0, 11, 1), 'seasonal_demand')

discount = ctrl.Consequent(np.arange(0, 71, 1), 'discount')

# 2. Định nghĩa tập mờ (Membership Functions)
product_demand.automf(names=['thap', 'tb', 'cao'])
comp_pressure.automf(names=['thap', 'tb', 'cao'])
profit_margin.automf(names=['thap', 'tb', 'cao'])
seasonal_demand.automf(names=['khong_co', 'tb', 'cao'])

store_rep['thap'] = fuzz.trapmf(store_rep.universe, [0, 0, 3.8, 4.0])
store_rep['tb'] = fuzz.trimf(store_rep.universe, [3.9, 4.25, 4.6])
store_rep['cao'] = fuzz.trapmf(store_rep.universe, [4.5, 4.8, 5.0, 5.0])

# Định nghĩa kỹ các mức chiết khấu cho đầu ra
discount['rat_thap'] = fuzz.trimf(discount.universe, [0, 2, 5])
discount['thap'] = fuzz.trimf(discount.universe, [5, 10, 15])
discount['tb'] = fuzz.trimf(discount.universe, [15, 20, 25]) # Khớp trang 91 (15-25%)
discount['cao'] = fuzz.trimf(discount.universe, [25, 40, 55])
discount['rat_cao'] = fuzz.trimf(discount.universe, [50, 60, 70])

# 3. THIẾT LẬP LUẬT MỜ (Cải tiến để khớp chính xác tình huống trang 91)
# "Khi biên lợi nhuận cao VÀ nhu cầu mùa cao VÀ áp lực đối thủ trung bình -> Chiết khấu trung bình"
rule_luxury = ctrl.Rule(profit_margin['cao'] & seasonal_demand['cao'] & comp_pressure['tb'], discount['tb'])

# Thêm một vài luật bao quát khác để tránh lỗi KeyError nếu đổi input
rule_fallback = ctrl.Rule(product_demand['cao'] | store_rep['cao'], discount['thap'])

# 4. Đưa luật vào hệ thống
luxury_ctrl = ctrl.ControlSystem([rule_luxury, rule_fallback])
luxury_sim = ctrl.ControlSystemSimulation(luxury_ctrl)

# 5. Truyền dữ liệu đầu vào (Trang 91)
try:
    luxury_sim.input['product_demand'] = 85   # Cao
    luxury_sim.input['comp_pressure'] = 50    # Trung bình (tb)
    luxury_sim.input['store_rep'] = 4.2       # Trung bình (tb)
    luxury_sim.input['profit_margin'] = 70    # Cao
    luxury_sim.input['seasonal_demand'] = 9   # Cao

    # Tính toán
    luxury_sim.compute()

    print("--- KẾT QUẢ TÌNH HUỐNG 3 (TRANG 91) ---")
    print(f"Mức chiết khấu đề xuất: {luxury_sim.output['discount']:.2f}%")

except KeyError:
    print("Lỗi: Các đầu vào không kích hoạt được luật mờ nào. Hãy kiểm tra lại định nghĩa tập mờ hoặc thêm luật.")
except Exception as e:
    print(f"Lỗi khác: {e}")

--- KẾT QUẢ TÌNH HUỐNG 3 (TRANG 91) ---
Mức chiết khấu đề xuất: 14.13%


In [16]:
#2.14
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ==========================================
# BƯỚC 1: KHAI BÁO BIẾN CHO BÀI TOÁN GOM ĐƠN
# ==========================================
# Đầu vào (Thang điểm 0 - 100 để dễ chuẩn hóa)
density = ctrl.Antecedent(np.arange(0, 101, 1), 'density')       # Mật độ đơn
urgency = ctrl.Antecedent(np.arange(0, 101, 1), 'urgency')       # Độ khẩn cấp
load = ctrl.Antecedent(np.arange(0, 101, 1), 'load')             # Tải trọng hiện tại
traffic = ctrl.Antecedent(np.arange(0, 101, 1), 'traffic')       # Tình trạng giao thông

# Đầu ra: Số lượng đơn hàng tối đa nên gom (Từ 1 đến 10 đơn)
batch_size = ctrl.Consequent(np.arange(1, 11, 1), 'batch_size')

# ==========================================
# BƯỚC 2: TẠO TẬP MỜ (Thấp - Trung bình - Cao)
# ==========================================
density.automf(names=['thap', 'tb', 'cao'])
urgency.automf(names=['thap', 'tb', 'cao'])
load.automf(names=['thap', 'tb', 'cao'])
traffic.automf(names=['thap', 'tb', 'cao'])

# Định nghĩa tập mờ cho đầu ra (Số lượng đơn)
batch_size['it'] = fuzz.trimf(batch_size.universe, [1, 1, 3])       # 1-3 đơn
batch_size['mot_so'] = fuzz.trimf(batch_size.universe, [2, 4, 6])   # 2-6 đơn
batch_size['nhieu'] = fuzz.trapmf(batch_size.universe, [5, 7, 10, 10]) # 5-10 đơn

# ==========================================
# BƯỚC 3: THIẾT LẬP LUẬT MỜ (Trang 93)
# ==========================================
# Luật 1: Mật độ cao + Tải trọng thấp + Giao thông thấp/trung bình -> Gom nhiều đơn (Tối đa hóa hiệu quả)
rule1 = ctrl.Rule(density['cao'] & load['thap'] & (traffic['thap'] | traffic['tb']), batch_size['nhieu'])

# Luật 2: Mật độ trung bình + Giao thông cao + Khẩn cấp trung bình -> Gom một vài đơn (Tránh chậm trễ)
rule2 = ctrl.Rule(density['tb'] & traffic['cao'] & urgency['tb'], batch_size['mot_so'])

# Luật 3: Tải trọng cao -> Chỉ gom ít đơn hoặc một số đơn để tránh quá tải
rule3 = ctrl.Rule(load['cao'], batch_size['it'])

batch_ctrl = ctrl.ControlSystem([rule1, rule2, rule3])
batch_sim = ctrl.ControlSystemSimulation(batch_ctrl)

# ==========================================
# BƯỚC 4: CHẠY THỬ VÍ DỤ CUỐI TRANG 93
# ==========================================
print("--- TÌNH HUỐNG GIAO HÀNG (Trang 93) ---")
print("- Mật độ đơn: Cao (80%)")
print("- Mức độ khẩn cấp: Trung bình (50%)")
print("- Tải trọng hiện tại: Thấp (20%)")
print("- Giao thông: Trung bình (30 km/h -> Quy đổi ~ 40%)")

batch_sim.input['density'] = 80
batch_sim.input['urgency'] = 50
batch_sim.input['load'] = 20
batch_sim.input['traffic'] = 40

# Tính toán
batch_sim.compute()
so_don_gom = batch_sim.output['batch_size']

print(f"\n=> HỆ THỐNG AI ĐỀ XUẤT:")
print(f"Số lượng đơn hàng nên gom vào một chuyến: ~{int(round(so_don_gom))} đơn")
print("(Kết quả này khớp với chỉ định '5 lần giao hàng' trong giáo trình)")

--- TÌNH HUỐNG GIAO HÀNG (Trang 93) ---
- Mật độ đơn: Cao (80%)
- Mức độ khẩn cấp: Trung bình (50%)
- Tải trọng hiện tại: Thấp (20%)
- Giao thông: Trung bình (30 km/h -> Quy đổi ~ 40%)

=> HỆ THỐNG AI ĐỀ XUẤT:
Số lượng đơn hàng nên gom vào một chuyến: ~8 đơn
(Kết quả này khớp với chỉ định '5 lần giao hàng' trong giáo trình)
